In [ ]:
"""

Title: "Scalable Neural Safety Certification via Monotonicity"
Code for: Code for Inductive vs. classic monotone barriers section of Appendix.
Author: Amirreza Alavi
Contact: seyedamirreza.alavi@colorado.edu
Affiliation: University of Colorado Boulder
Date: 04/2026

"""

In [ ]:
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers, constraints, initializers
import matplotlib.pyplot as plt
from numpy import linalg as LA
from scipy import stats
from functools import partial
%matplotlib inline
import numpy as np
from scipy.optimize import fsolve
from tensorflow.keras.layers import Input, Dense, Concatenate, Add
from tensorflow.keras import Model
from tensorflow.keras import regularizers
from tensorflow.keras.models import load_model
import time
from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D  
from math import inf
import heapq
from dataclasses import dataclass
import matplotlib.patches as mpatches
import matplotlib as mpl
from matplotlib.patches import Rectangle

mpl.rcParams["text.usetex"] = True
mpl.rcParams["font.family"] = "serif"

In [ ]:
# System Definition

def phi(x):
    return tf.constant(10,dtype=tf.float32)*(1-tf.exp(-0.1*tf.constant(x,dtype=tf.float32)))

def sigma(x):
    return  tf.maximum(0,tf.constant(10,dtype=tf.float32)*(1- tf.pow(x/tf.constant(50,dtype=tf.float32),3)))


def f_m(q):
    q = tf.convert_to_tensor(q, dtype=tf.float32)
    tau = tf.constant(0.5, dtype=tf.float32)

    x1 = q[..., 0]
    x2 = q[..., 1]

    y1 = x1 + tau * (8.3 - tf.minimum(phi(x1), sigma(x2)))
    y2 = x2 + tau * (tf.minimum(phi(x1), sigma(x2)) - phi(x2))

    y = tf.stack([y1, y2], axis=-1)
    return tf.clip_by_value(y, 0.0, 50)


def simulate(q0_batch, steps):
    q = tf.convert_to_tensor(q0_batch, tf.float32)
    traj = [q]
    for _ in range(steps):
        q = f_m(q)
        traj.append(q)
    return tf.stack(traj, axis=1).numpy()

In [ ]:
# State set, Initial set and Unsafe set visualization along with sample data evaluation


mpl.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "axes.linewidth": 1.0,
})

x_in_bar   = tf.constant([[10., 10.]], dtype=tf.float32)  
x_us1_und  = tf.constant([[0., 40.]],  dtype=tf.float32)  
x_us2_und  = tf.constant([[40., 0.]],  dtype=tf.float32)  
x_s_und    = tf.constant([[0., 0.]],   dtype=tf.float32)  
x_s_bar    = tf.constant([[40., 40.]], dtype=tf.float32)  


X = tf.concat([x_in_bar, x_us1_und, x_us2_und, x_s_und, x_s_bar], axis=0) 


x_s_bar_next = f_m(x_s_bar)  


plt.figure(figsize=(6, 6))
ax = plt.gca()


pad = 1
xmin, xmax = -pad, 50 + pad
ymin, ymax = -pad, 50 + pad


ax.add_patch(Rectangle((0, 0), 10, 10, facecolor="blue", edgecolor="none", alpha=0.4))


ax.add_patch(Rectangle((0, 40), 40, 10, facecolor="red", edgecolor="none", alpha=0.4))
ax.add_patch(Rectangle((40, 0), 10, 50, facecolor="red", edgecolor="none", alpha=0.4))


ax.add_patch(Rectangle((0, 0), 50, 50, fill=False, linewidth=1.5, alpha=0.8))


plt.scatter(X[0, 0].numpy(), X[0, 1].numpy(), marker="s", s=50, label=r"$\bar{x}_{in}$", c='green')
plt.scatter(X[1, 0].numpy(), X[1, 1].numpy(), marker="s", s=50, label=r"$\underline{x}^1_{us}$", c='red')
plt.scatter(X[2, 0].numpy(), X[2, 1].numpy(), marker="s", s=50, label=r"$\underline{x}^1_{us}$", c='red')
plt.scatter(X[3, 0].numpy(), X[3, 1].numpy(), marker="s", s=50, label=r"$\underline{x}{s}$", c='blue')
plt.scatter(X[4, 0].numpy(), X[4, 1].numpy(), marker="s", s=50, label=r"$\bar{x}_s$", c="#1f99b2")

x0 = x_s_bar[0].numpy()
x1 = x_s_bar_next[0].numpy()

plt.plot([x0[0], x1[0]], [x0[1], x1[1]], linewidth=2.0, alpha=0.9)
plt.scatter([x1[0]], [x1[1]], marker="^", s=70, label=r"$f(\bar{x}_s)$", c="#1f77b4")


plt.xlim(xmin, xmax)
plt.ylim(ymin, ymax)
ax.set_aspect("equal", "box")

plt.xlabel(r"$q_1$")
plt.ylabel(r"$q_2$")
#plt.title(r"Evaluating one-step dynamics on extreme grid points")

plt.legend(loc="best", frameon=False, fontsize=14)
plt.tight_layout()
plt.savefig("grideval.pdf", bbox_inches="tight")
plt.show()


In [ ]:
def plot_rectangles(lows, highs, *, fill=False, annotate=True, ax=None, alpha=0.15, lw=1.8):

    lows  = np.asarray(lows,  dtype=float)
    highs = np.asarray(highs, dtype=float)
    assert lows.ndim == highs.ndim == 2 and lows.shape == highs.shape and lows.shape[1] == 2
    

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    n = lows.shape[0]
    for i in range(n):
        (x0, y0), (x1, y1) = lows[i], highs[i]
        w, h = x1 - x0, y1 - y0
        rect = mpatches.Rectangle((x0, y0), w, h,
                                  fill=fill,
                                  linewidth=lw,
                                  edgecolor='black',
                                  facecolor='C0' if fill else 'none',
                                  alpha=alpha if fill else 1.0)
        ax.add_patch(rect)
        if annotate:
            ax.text(x0 + w/2, y0 + h/2, f"{i}", ha='center', va='center', fontsize=9)

    x_min = float(np.min(lows[:, 0])); x_max = float(np.max(highs[:, 0]))
    y_min = float(np.min(lows[:, 1])); y_max = float(np.max(highs[:, 1]))
    dx = (x_max - x_min) * 0.02 or 1.0
    dy = (y_max - y_min) * 0.02 or 1.0
    ax.set_xlim(x_min - dx, x_max + dx)
    ax.set_ylim(y_min - dy, y_max + dy)

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.set_title('Hyper-rectangles')
    ax.grid(True, linestyle=':', linewidth=0.7)
    return ax


In [ ]:
# Sampled points for a coarse gridding and system inductive evaluation

N=2
k=100

xin_bar = np.ones((1,N))*10
xin_und = np.zeros((1,N))
xuns_und = 40 * np.eye(N, dtype=np.float32)
xuns_bar = np.ones((1,N))*50
xi_bar = tf.ones((1, N), dtype=tf.float32) * 40.0
xi_und = tf.zeros((1, N), dtype=tf.float32)

def f_m_0km1(q, k):
    q = tf.convert_to_tensor(q, dtype=tf.float32)
    if q.shape.rank == 1:
        q = tf.expand_dims(q, 0)     
    B, N = q.shape
    traj = tf.TensorArray(tf.float32, size=k, element_shape=(B, N))
    for k in tf.range(k):
        traj = traj.write(k, q)      
        q = f_m(q)                  
    return traj.stack() 


def f_m_k(q, k):
    x = tf.convert_to_tensor(q, dtype=tf.float32)
    for _ in tf.range(k):
        x = f_m(x)
    return x

f_m_0km1_xin_bar = tf.squeeze(f_m_0km1(xin_bar,k))
f_m_0km1_xi_und = tf.squeeze(f_m_0km1(xi_und,k), axis=1)
f_m_k_xi_bar = f_m_k(xi_bar,k)

In [ ]:
# Barrier Neural Network Definition

max_layer = layers.Lambda(lambda x: tf.reduce_max(x, axis=1, keepdims=True))

modelB = tf.keras.Sequential()
modelB.add(layers.Input(shape=(N,)))
modelB.add(max_layer) 
modelB.add(layers.Dense(5, use_bias=True, activation=tf.keras.layers.LeakyReLU(alpha=0.1),
                        kernel_constraint=tf.keras.constraints.NonNeg()))
modelB.add(layers.Dense(1, use_bias=True, activation=None,
                        kernel_constraint=tf.keras.constraints.NonNeg()))

In [ ]:
# Training Monotone Neural Inductive Barrier

epochs = 100000
opt1 = tf.keras.optimizers.Adam(1e-2)
t1 = time.perf_counter()

gamma = 5
eta = 10


for i in range(epochs):
    with tf.GradientTape() as tape0, tf.GradientTape() as tape1:

        loss_B = tf.reduce_sum(tf.nn.relu(tf.subtract(1.0,1.0)))

        loss_B += tf.square(tf.nn.relu(tf.reduce_max(modelB(f_m_0km1_xin_bar))- gamma))
        loss_B += tf.square(tf.nn.relu(eta-tf.reduce_min(modelB(xuns_und))))
        loss_B += tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_m_0km1_xi_und)) - 9.5)))  * (tf.nn.relu(modelB(f_m_k_xi_bar) -  9.5))

                                
        grad_B = tape0.gradient(loss_B, modelB.trainable_variables)
                
        if tf.reduce_max(modelB(f_m_0km1_xin_bar)) < gamma and tf.reduce_min(modelB(xuns_und)) > eta and  tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_m_0km1_xi_und)) - 9.9)))  * (tf.nn.relu(modelB(f_m_k_xi_bar) -  9.9)) == 0   and i>1:
            print('Safety Verified')
            break
            
        if loss_B!=0:
            opt1.apply_gradients(zip(grad_B, modelB.trainable_variables),)
            
        if i%5 == 1:
            print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(f_m_0km1_xin_bar)).numpy():.8f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(xuns_und)).numpy():.8f}, "
            f"Third condition = {tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_m_0km1_xi_und)) - 9.0)))  * (tf.nn.relu(modelB(f_m_k_xi_bar) -  9.9)).numpy():}, "
            f"LossB = {loss_B}"
            )


    
t2 = time.perf_counter()
tot = t2 - t1
print(f'Total runtime: {tot}s')
print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(f_m_0km1_xin_bar)).numpy():.8f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(xuns_und)).numpy():.8f}, "
            f"Third condition = {tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_m_0km1_xi_und)) - 9.9)))  * (tf.nn.relu(modelB(f_m_k_xi_bar) -  9.9)).numpy():}, "
            f"LossB = {loss_B}"
            )

In [ ]:
# Sub-level set of ordinary barrier on a coarse gridding 

mpl.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "axes.linewidth": 1.0,
})

x = np.linspace(0, 50, 200)
y = np.linspace(0, 50, 200)
z1, z2 = np.meshgrid(x, y)
z = np.stack([z1, z2], axis=-1).reshape(-1, 2)

def max_modelB_over_traj(z, K):
    traj = f_m_0km1(z, K)                       
    traj_flat = tf.reshape(traj, (-1, traj.shape[-1]))  
    outputs = modelB(traj_flat)                 
    outputs = tf.reshape(outputs, (K, -1))      
    return tf.reduce_max(outputs, axis=0)       

k = 100
modelB_output = max_modelB_over_traj(z, k)
condition = modelB_output.numpy().reshape(z1.shape)

etaa = 9.99


plt.figure(figsize=(6, 6))
ax = plt.gca()


ax.add_patch(Rectangle((0, 0), 50, 50, fill=False, linewidth=1.5, alpha=0.8))

init_rect = Rectangle((0, 0), 10, 10, facecolor="blue", edgecolor="none", alpha=0.4)
ax.add_patch(init_rect)

unsafe_top = Rectangle((0, 40), 40, 10, facecolor="red", edgecolor="none", alpha=0.4)
unsafe_right = Rectangle((40, 0), 10, 50, facecolor="red", edgecolor="none", alpha=0.4)
ax.add_patch(unsafe_top)
ax.add_patch(unsafe_right)


ax.contourf(z1, z2, condition, levels=[-np.inf, etaa], colors=["lightgray"], alpha=0.3)
ax.contour(z1, z2, condition, levels=[etaa], colors=["black"], linewidths=1.2, alpha=1.0)


ax.set_xlabel(r"$x_1$", fontsize=16)
ax.set_ylabel(r"$x_2$", fontsize=16)
ax.set_xlim(0, 50)
ax.set_ylim(0, 50)
ax.set_aspect("equal", adjustable="box")


ax.set_xticks(np.arange(0, 51, 10))
ax.set_yticks(np.arange(0, 51, 10))

init_patch = mpatches.Patch(color="blue", alpha=0.15, label=r"Initial set ($\mathcal{X}_0$)")
unsafe_patch = mpatches.Patch(color="red", alpha=0.12, label=r"Unsafe set ($\mathcal{X}_u$)")
gray_patch = mpatches.Patch(color="lightgray", alpha=0.5, label=r"$\max_{ 0\le k<K} B(f^k(x)) \le \delta$")
 
ax.legend(handles=[init_patch, unsafe_patch, gray_patch], loc="best", fontsize=12, frameon=True)
plt.tight_layout()
#plt.savefig("indtraf.pdf", bbox_inches="tight")
plt.show()
